 # EirGrid Wind Power Forecasting — Big Data Pipeline
  **Student:** Sebastian Briones **Student ID:** 20206004
  **Module:** Big Data Storage & Processing
  **Dataset:** EirGrid Smart Grid Dashboard (2014–2026)
  **Objective:** Process raw EirGrid data with Apache Spark and prepare features for LSTM forecasting

In [6]:
 # Adapted from: Tutorial 5 (a).ipynb — Cell 5 (sc = spark.sparkContext)            
  # and Tutorial 5 (c).ipynb — Cell 3 (sc = spark.sparkContext, import warnings)
  # SparkSession.builder is el patrón estándar cuando pyspark no lo crea             
  #automáticamente                                                                    
                                                                                     
from pyspark.sql import SparkSession                                               
from pyspark.sql import functions as F          # Tutorial 4 - NN using            
  #Pyspark.ipynb — Cell 6                                                  
from pyspark.sql.types import DoubleType        # Tutorial 4 - NN using            
  #Pyspark.ipynb — Cell 2 (from pyspark.sql.types import *)                
from pyspark.sql.window import Window           # Spark API estándar (no está en   
  #tutoriales directamente)                                                         
                                                                                     
import warnings                                 # Tutorial 5 (c).ipynb — Cell 4
warnings.filterwarnings("ignore")               # Tutorial 5 (c).ipynb — Cell 4    
                                                                                 
spark = SparkSession.builder.appName("EirGrid_CA1_2026004").config("spark.sql.shuffle.partitions", "8").getOrCreate()                           
                                                                                     
spark.sparkContext.setLogLevel("WARN")                                             
print(f"Spark version: {spark.version}")                                           
print(f"App name: {spark.sparkContext.appName}")   

Spark version: 4.1.2
App name: PySparkShell


26/10/03 21:43:44 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [7]:
df_raw = spark.read.csv(                                                           
      "hdfs:///user/sbriones/ca1/eirgrid_raw/eirgrid_ALL_*.csv",                     
      header=True,                                                                   
      inferSchema=True
)
print(f"Total rows: {df_raw.count():,}")                                           
print(f"Columns: {df_raw.columns}")                                                
df_raw.printSchema()                                                               
df_raw.show(5, truncate=False) 

26/10/03 21:45:00 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: hdfs:///user/sbriones/ca1/eirgrid_raw/eirgrid_ALL_*.csv.
java.io.FileNotFoundException: File does not exist: hdfs:/user/sbriones/ca1/eirgrid_raw/eirgrid_ALL_*.csv
	at org.apache.hadoop.hdfs.DistributedFileSystem$29.doCall(DistributedFileSystem.java:1842)
	at org.apache.hadoop.hdfs.DistributedFileSystem$29.doCall(DistributedFileSystem.java:1835)
	at org.apache.hadoop.fs.FileSystemLinkResolver.resolve(FileSystemLinkResolver.java:81)
	at org.apache.hadoop.hdfs.DistributedFileSystem.getFileStatus(DistributedFileSystem.java:1850)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(Resolve

Total rows: 897,468
Columns: ['EffectiveTime', 'SYSTEM_DEMAND', 'GEN_EXP', 'WIND_ACTUAL', 'CO2_INTENSITY', 'CO2_EMISSIONS', 'solar_actual', 'solar_forecast', 'INTER_EWIC', 'INTER_MOYLE', 'INTER_NET', 'INTER_GRNLK']
root
 |-- EffectiveTime: string (nullable = true)
 |-- SYSTEM_DEMAND: string (nullable = true)
 |-- GEN_EXP: string (nullable = true)
 |-- WIND_ACTUAL: string (nullable = true)
 |-- CO2_INTENSITY: string (nullable = true)
 |-- CO2_EMISSIONS: string (nullable = true)
 |-- solar_actual: string (nullable = true)
 |-- solar_forecast: string (nullable = true)
 |-- INTER_EWIC: string (nullable = true)
 |-- INTER_MOYLE: string (nullable = true)
 |-- INTER_NET: string (nullable = true)
 |-- INTER_GRNLK: string (nullable = true)

+-------------------+-------------+-------+-----------+-------------+-------------+------------+--------------+----------+-----------+---------+-----------+
|EffectiveTime      |SYSTEM_DEMAND|GEN_EXP|WIND_ACTUAL|CO2_INTENSITY|CO2_EMISSIONS|solar_actual|solar

In [9]:
df_raw.show()

+-------------------+-------------+-------+-----------+-------------+-------------+------------+--------------+----------+-----------+---------+-----------+
|      EffectiveTime|SYSTEM_DEMAND|GEN_EXP|WIND_ACTUAL|CO2_INTENSITY|CO2_EMISSIONS|solar_actual|solar_forecast|INTER_EWIC|INTER_MOYLE|INTER_NET|INTER_GRNLK|
+-------------------+-------------+-------+-----------+-------------+-------------+------------+--------------+----------+-----------+---------+-----------+
|2014-01-01 00:00:00|       3758.0| 3390.0|     1386.0|        383.0|       1346.0|        NULL|          NULL|      NULL|       NULL|     NULL|       NULL|
|2014-01-01 00:15:00|       3724.0| 3359.0|     1348.0|        383.0|       1335.0|        NULL|          NULL|      NULL|       NULL|     NULL|       NULL|
|2014-01-01 00:30:00|       3666.0| 3301.0|     1272.0|        383.0|       1314.0|        NULL|          NULL|      NULL|       NULL|     NULL|       NULL|
|2014-01-01 00:45:00|       3611.0| 3246.0|     1273.0|   